# Lab 2 — Phát hiện ảnh GAN giả mạo: 3 GAN + Grad-CAM (Colab GPU)

**Repo**: https://github.com/nmnhut-it/math-for-ai

**Trước khi chạy**: Runtime → Change runtime type → **GPU** (T4 free / L4 Pro / A100).

Pipeline (`Run all` 1 lần là xong, ~15-20 phút trên L4):

1. cGAN-MNIST (in-house, MLP) → TinyCNN scratch ~99% + Grad-CAM (jitter pixel-level)
2. PGAN-DTD scratch → TexCNN ~62% (narrative "PGAN khó")
3. PGAN-DTD transfer → ResNet18 ~99% (cứu narrative — không phải PGAN khó, là detector yếu)
4. BigGAN-128 transfer → ResNet18 ~99% (3rd benchmark, class-conditional)
5. Grad-CAM trên ResNet18 (cả PGAN + BigGAN) → giải thích vùng nào kích hoạt detector

## 1. Setup runtime + clone repo

In [ ]:
import torch, os
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('!!! Khong co GPU — bat GPU runtime truoc khi chay !!!')

In [ ]:
# Robust against nested re-clone: dung absolute path, cd ve home truoc.
import os
ROOT = '/content/math-for-ai'
%cd /content
if not os.path.exists(ROOT):
    !git clone https://github.com/nmnhut-it/math-for-ai.git
else:
    !cd {ROOT} && git pull
%cd {ROOT}/lab2_cnn
!pwd

In [ ]:
!pip install -q pytorch-pretrained-biggan

## 2. cGAN-MNIST (in-house) — TinyCNN scratch + Grad-CAM

Train cGAN 30 epoch (cache `../lab2/output/cG_final.pth` đã có, không train lại). TinyCNN 105 k params train từ đầu trên 10k+10k. Grad-CAM trên `conv2` cho ảnh real + fake. Kèm phân tích định lượng: lệch khỏi -1 ở vùng nền tuyệt đối, Pearson(high-freq, attention).

~30s training + ~5s Grad-CAM.

In [ ]:
!python lab2_cnn.py

In [ ]:
!python gradcam.py

## 3. PGAN-DTD — TexCNN scratch (baseline narrative)

Sample 1500 PGAN-DTD fakes + 1500 DTD reals. TexCNN 564 k params train từ đầu. Kết quả ~62.5% — đây là baseline để chứng minh "small CNN scratch không đủ cho modern GAN". Sẽ được phản bác bằng Section 4 (ResNet18 transfer trên cùng PGAN).

~2-3 phút (sample PGAN ~1.5 phút, train ~30s).

In [ ]:
!python lab2_cnn_pgan.py

## 4. PGAN-DTD — ResNet18 transfer (tách confound)

Cùng PGAN data nhưng dùng ResNet18 pretrained ImageNet (11 M params), transfer 2 phase:
- Phase 1 (3 epoch, lr=1e-3): freeze backbone, chỉ train Linear(512→2)
- Phase 2 (12 epoch, lr=1e-4): unfreeze layer4 + finetune

Kỳ vọng ~98% — cùng PGAN nhưng nhảy +35 điểm nhờ transfer learning. Phase 1 alone đạt ~88% → chứng minh ImageNet features đã encode "natural image manifold" sẵn.

~5-6 phút.

In [ ]:
!python lab2_cnn_pgan_resnet.py

## 5. BigGAN-128 + Imagenette — ResNet18 transfer

Benchmark thứ 3 với class-conditional GAN top-tier 2018 (BigGAN, DeepMind). Sample 2500 fakes (1000 lớp ImageNet ngẫu nhiên, truncation 0.4) + 2500 reals từ Imagenette-160.

Cùng ResNet18 transfer pipeline như PGAN. Kỳ vọng ~99%.

**Lần đầu**: download ~340 MB BigGAN weights + 94 MB Imagenette tarball.

~6-7 phút.

In [ ]:
!python lab2_cnn_biggan.py

## 6. Grad-CAM trên ResNet18 — explainable AI cho 2 modern GAN

Inference trên 4 real + 4 fake từ val set của mỗi GAN. Target layer = `model.layer4` (last conv, feature map 4×4 cho input 128×128). Heatmap upsample về 128×128, overlay trên ảnh gốc.

Mục đích: kiểm chứng giả thuyết "ResNet18 detect dựa trên distribution shift của features ImageNet". Vùng đỏ = nơi feature vector lệch xa khỏi natural-image manifold mà ImageNet encode.

~30s (chỉ inference).

In [ ]:
!python gradcam_resnet.py

## 7. Hiển thị tất cả kết quả inline

In [ ]:
from IPython.display import Image, display
import os

def show(label, txt_path, img_paths):
    print('=' * 70)
    print(f'=== {label} ===')
    print('=' * 70)
    if os.path.exists(txt_path):
        with open(txt_path) as f:
            print(f.read())
    else:
        print(f'(missing {txt_path})')
    for p in img_paths:
        if os.path.exists(p):
            display(Image(p))
        else:
            print(f'(missing {p})')

show('1. cGAN-MNIST + TinyCNN scratch', 'output/results.txt',
     ['output/confusion_matrix.png',
      'output/gradcam_overlay.png',
      'output/gradcam_mean.png',
      'output/gradcam_corr.png'])

In [ ]:
show('2. PGAN-DTD + TexCNN scratch (baseline)', 'output/results_pgan.txt',
     ['output/confusion_matrix_pgan.png'])

In [ ]:
show('3. PGAN-DTD + ResNet18 transfer', 'output/results_pgan_resnet.txt',
     ['output/confusion_matrix_pgan_resnet.png',
      'output/gradcam_pgan_resnet.png'])

In [ ]:
show('4. BigGAN-128 + Imagenette + ResNet18 transfer',
     'output/results_biggan.txt',
     ['output/biggan_samples.png',
      'output/confusion_matrix_biggan.png',
      'output/gradcam_biggan.png'])

## 8. (Tùy chọn) Download kết quả về local

```python
from google.colab import files
import os
for f in [
    'output/results.txt',
    'output/results_pgan.txt',
    'output/results_pgan_resnet.txt',
    'output/results_biggan.txt',
    'output/confusion_matrix.png',
    'output/confusion_matrix_pgan.png',
    'output/confusion_matrix_pgan_resnet.png',
    'output/confusion_matrix_biggan.png',
    'output/gradcam_overlay.png',
    'output/gradcam_mean.png',
    'output/gradcam_corr.png',
    'output/gradcam_pgan_resnet.png',
    'output/gradcam_biggan.png',
    'output/biggan_samples.png',
    'output/cnn_best.pth',
    'output/cnn_pgan_best.pth',
    'output/cnn_pgan_resnet_best.pth',
    'output/cnn_biggan_resnet_best.pth',
]:
    if os.path.exists(f):
        files.download(f)
```